# *Breast Cancer Detection Using Deep Learning (Fuzzy)*

# 01 - Data Exploration

Purpose:
- Load raw breast thermogram images
- Visualize sample images
- Inspect image size, channels and intensity range
- Verify dataset structure

Dataset(s):
- DMR-IR
- DBT-TU-JU

In [ ]:
import sys, os
import numpy as np
import pandas as pd
from PIL import Image
# from pathlib import Path
import matplotlib.pyplot as plt

In [ ]:
sys.path.append(os.path.abspath(".."))

In [ ]:
from src.utils import section

## STAGE 0: PLOTTING

In [ ]:
from src.plotting import DatasetPlot, DatasetOrganization
from src.utils import base_path, dmr_ir_o, dbt_tu_ju, dmr_ground_truth, dbt_tu_ju_ground_truth

In [ ]:
DMR_IR_PATH = base_path + dmr_ir_o
DBT_TU_JU_PATH = base_path + dbt_tu_ju

### DMR-IR DATASET PLOT

In [ ]:
plotter_dmr = DatasetPlot(3, DMR_IR_PATH)
plotter_dmr.main()

### DBT-TU-JU DATASET PLOT

In [ ]:
df = DatasetOrganization(DBT_TU_JU_PATH)

print(df.head())
print("Total images:", len(df))
print("Total patients:", df["patient_id"].nunique())

In [ ]:
img = Image.open(df.iloc[0]["image_path"])

plt.imshow(img, cmap="gray")
plt.axis("off")

### GROUND TRUTH - DBT-TU-JU

In [ ]:
plotter_gt_tu = DatasetPlot(3, dbt_tu_ju_ground_truth)
plotter_gt_tu.main()

### GROUND TRUTH - DMR

In [ ]:
plotter_gt_dmr = DatasetPlot(4, dmr_ground_truth)
plotter_gt_dmr.main()

## STAGE 1: PRE-PROCESSING

In [ ]:
from src.preprocessing import StorageConfig, run_preprocessing
from src.utils import PRE_CFG, SCH_CFG, base_path, dmr_ir_o, load_preprocessing_results, preprocessed_results_path

In [ ]:
DMR_IR_PATH = base_path + dmr_ir_o

image_files = sorted([
    f for f in os.listdir(DMR_IR_PATH)
    if f.lower().endswith(('.jpg', '.png', '.jpeg'))
])
print(f'Found {len(image_files)} images:')
for i, f in enumerate(image_files):
    print(f'  [{i}] {f}')

In [ ]:
IMAGE_INDEX = 6   # ← change this to try a different image
IMAGE_PATH  = os.path.join(DMR_IR_PATH, image_files[IMAGE_INDEX])

In [ ]:
preprocessing_results = run_preprocessing(
    image_path=IMAGE_PATH,
    cfg=PRE_CFG,
    visualize=True,
    save_results=True,
    storage_config=StorageConfig(output_dir=preprocessed_results_path)
)

# PREPROCESSING RESULTS
print(f"\nResults saved to: {preprocessing_results['run_dir']}")
print(f"p_b saved at: {preprocessing_results['saved_paths']['pb']}")

In [ ]:
# lOADING PRE-PROCESSING RESULTS FOR NEXT STEPS:
preprocessing_results_loaded = load_preprocessing_results(preprocessing_results['run_dir'])
pb = preprocessing_results_loaded['pb']
image_name = preprocessing_results_loaded['base_name']
print(image_name)

## STAGE 1: SCH-CS

In [ ]:
from src.sch_cs import run_schcs, SchCsStorageConfig
from src.utils import schcs_results_path, load_schcs_results

In [ ]:
schcs_result = run_schcs(
    pb=pb,
    cfg=SCH_CFG,
    visualize=True,
    image_name=image_name,
    save_results=True,
    storage_config=SchCsStorageConfig(output_dir=schcs_results_path)
)

# SCH-CS RESULTS
print(f"\nResults saved to: {schcs_result['run_dir']}")
print(f"Binary image (for Level Set) saved at: {schcs_result['saved_paths']['binary_image']}")

In [ ]:
# GLOBAL DICTIONARY
schcs_runs = {}

# TRACKING DICTIONARY
schcs_runs[image_name] = {
    'run_dir': schcs_result['run_dir'],
    'timestamp': schcs_result['timestamp'],
    'num_regions': len(schcs_result['sr_regions'])
}

# SCH-CS RESULTS FOR LEVEL SET
schcs_results_loaded  = load_schcs_results(schcs_runs[image_name]['run_dir'])
p_th_b = schcs_results_loaded['binary_image']  # This is p_th_b for Eq 17
# GET N_SR FROM METADATA
n_sr = schcs_results_loaded['metadata']['num_sr_regions']

## STAGE 2: CHM Bias Correction

In [ ]:
from src.utils import chm_corrected_results_path
from src.chm_bias_correction import CHMConfig, run_chm_pipeline

In [ ]:
section("CHM Bias Field Correction - Stage 2")

chm_bias_config = CHMConfig(
    save_visualization=True,
    show_visualization=True,
    output_dir=chm_corrected_results_path
)

chm_results = run_chm_pipeline(pb, chm_bias_config, image_name=image_name)

print("p_bar Saved here:", chm_results["run_dir"])

## STAGE 2: LEVEL SET INITIALIZATION

In [ ]:
from src.utils import phi_initialized_result_path
from src.level_set_initialization import PhiInitConfig, run_phi_init_pipeline

In [ ]:
section("Level Set Function φ Initialization - Stage 2 Step 2")

# Configuration
level_set_init_config = PhiInitConfig(
    inside_value=4.0,
    outside_value=-4.0,
    save_visualization=True,
    show_visualization=True,
    save_phi_array=True,
    output_dir=phi_initialized_result_path
)

# Run pipeline
phi_init_results = run_phi_init_pipeline(
    preprocessed_image=preprocessing_results['pb'],
    sr_regions=schcs_result['sr_regions'],
    config=level_set_init_config,
    image_name=image_name
)

phi = phi_init_results["phi"]

print("\n--- Verification ---")
print(f"phi shape: {phi.shape}")
print(f"phi dtype: {phi.dtype}")
print(f"phi values: {np.unique(phi)}   ← should be [-4. +4.]")
print(f"Saved in: {phi_init_results['run_dir']}")

## STAGE 2: LEVEL SET ITERATION

In [ ]:
from src.utils import level_set_iterated_results_path
from src.level_set_iteration import run_level_set, LevelSetConfig, run_level_set_per_sr, get_segmented_sr_with_fallback

In [ ]:
# Configuration
level_set_config = LevelSetConfig(
    alpha1=1.0,
    alpha2=1.0,
    theta=0.2,
    epsilon=1.5,
    dt=1.0,
    t_stop=0.01,
    max_iterations=500,
    verbose_every=50,
    output_dir=level_set_iterated_results_path
)

os.makedirs(level_set_config.output_dir, exist_ok=True)

In [ ]:
# ──────────────────────────────────────────────────────────
# CHOOSE WHICH METHOD TO USE
# ──────────────────────────────────────────────────────────

# Option A: Use the NEW per-SR processing (RECOMMENDED for multiple SRs)
# Option B: Use the ORIGINAL all-at-once processing (fine for single SR)

use_per_sr_processing = True  # ← Set to False to use original method

if use_per_sr_processing:
    print("\n" + "=" * 60)
    print("USING PER-SR LEVEL SET PROCESSING")
    print("=" * 60)

    # Run level set separately on each SR region
    level_set_iteration_results = run_level_set_per_sr(
        p_bar=chm_results['p_bar'],
        sr_regions=schcs_result['sr_regions'],
        image_name=image_name,
        preprocessed_img=pb,
        config=level_set_config, # Use new variable name
        margin=20,              # Padding around each SR crop
        verbose=True,
        do_visualize=True,
        do_save=True
    )

    # Get the raw segmentation from per-SR processing
    raw_segmented_sr = level_set_iteration_results['segmented_sr']

else:
    print("\n" + "=" * 60)
    print("USING ORIGINAL ALL-AT-ONCE LEVEL SET")
    print("=" * 60)

    # Original method — runs on entire image at once
    level_set_iteration_results = run_level_set(
        p_bar=chm_results['p_bar'],
        phi_init=phi,
        n_sr=n_sr,
        image_name=image_name,
        preprocessed_img=pb,
        config=level_set_config,
        verbose=True,
        do_visualize=True,
        do_save=True
    )

    raw_segmented_sr = level_set_iteration_results['segmented_sr']

# ──────────────────────────────────────────────────────────
# VALIDATING SEGMENTATION
# ──────────────────────────────────────────────────────────

# Apply validation and fallback logic
final_segmented_sr, source = get_segmented_sr_with_fallback(
    level_set_result={'segmented_sr': raw_segmented_sr},
    sr_regions=schcs_result['sr_regions'],
    image_shape=pb.shape
)

print(f"\n[Final] Using segmentation from: {source.upper()}")
print(f"[Final] SR pixels: {int(final_segmented_sr.sum())}")

# ──────────────────────────────────────────────────────────
# UPDATE THE RESULTS DICTIONARY
# ──────────────────────────────────────────────────────────

# Replace the segmented_sr with the validated/final version
level_set_iteration_results['segmented_sr'] = final_segmented_sr
level_set_iteration_results['segmentation_source'] = source

print(f"\nResults ready for SR Split stage.")

In [ ]:
# ── Optional: Visualize DLPE vs Fallback ────────────────────────────────────
if source == 'schcs_fallback':
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(pb, cmap='gray')
    axes[0].set_title('Original p_b')
    axes[0].axis('off')

    axes[1].imshow(pb, cmap='gray')
    axes[1].contour(raw_segmented_sr, levels=[0.5], colors=['red'], linewidths=1.5)
    axes[1].set_title(f'DLPE (REJECTED)\n{int(raw_segmented_sr.sum())}px')
    axes[1].axis('off')

    axes[2].imshow(pb, cmap='gray')
    axes[2].contour(final_segmented_sr, levels=[0.5], colors=['green'], linewidths=1.5)
    axes[2].set_title(f'SCH-CS Fallback (USED)\n{int(final_segmented_sr.sum())}px')
    axes[2].axis('off')

    plt.suptitle('⚠️ DLPE Failed Validation — Using SCH-CS Fallback',
                 fontweight='bold', color='darkred')
    plt.tight_layout()
    plt.show()

## STAGE 3: SR SEGMENTATION

In [ ]:
from src.utils import segmentation_results_path
from src.sr_segmentation import SRSplitConfig, run_sr_split_pipeline

In [ ]:
section("SR Segmentation - Stage 3 Step 1")

sr_split_config = SRSplitConfig(
    save_visualization=True,
    show_visualization=True,
    output_dir=segmentation_results_path
)

sr_split_results = run_sr_split_pipeline(
    preprocessed_img=pb,
    segmented_sr=level_set_iteration_results['segmented_sr'],
    config=sr_split_config,
    image_name=image_name,
    # OPTIONAL
    # centre_col_override=None
    centre_col_override=200
)

print("Saved at:", sr_split_results["run_dir"])

## STAGE 3: FEATURE EXTRACTION

In [ ]:
from src.utils import extracted_features_results_path
from src.features_extraction import FeatureConfig, run_feature_pipeline

In [ ]:
section("Feature Extraction - Stage 3 Step 2")

feature_names = (
    [f"H{i+1}" for i in range(14)] +
    [f"Hu{i+1}" for i in range(7)]
)

feature_config = FeatureConfig(
    output_dir=extracted_features_results_path,
    save_visualization=True,
    show_visualization=True
)

feature_extraction_results = run_feature_pipeline(
    preprocessed_img=pb,
    sr_left=sr_split_results['sr_left'],
    sr_right=sr_split_results['sr_right'],
    config=feature_config,
    image_name=image_name
)

print("Saved at:", feature_extraction_results["run_dir"])

## STAGE 3: ASYMMETRIC FEATURE VECTOR

In [ ]:
from src.utils import asymmetric_vector_results_path
from src.asymmetry_vector import run_asymmetry_pipeline

In [ ]:
asymmetry_results = run_asymmetry_pipeline(
    f_v_left=feature_extraction_results['f_v_left'],
    f_v_right=feature_extraction_results['f_v_right'],
    image_name=image_name,
    base_output_dir=asymmetric_vector_results_path,
    label=f"Patient_{image_name}",
    save_visualization=True,
    show_visualization=True
)

F = asymmetry_results["F"]

print(f"\nF shape: {F.shape}")
print(f"Saved in: {asymmetry_results['run_dir']}")

## STAGE 3: FANN CLASSIFIER

In [ ]:
from src.fann_classifier import build_fann, run_fann_5fold, compute_metrics, visualize_fann_results, visualize_single_image_classification

In [ ]:
# To make a single prediction, we'll use the model and scaler from the first fold of the cross-validation
# This is for demonstration; in a real application, you might use an ensemble or a model retrained on all data.

if 'fann_results' in locals() and fann_results['fold_models']:
    demo_model = fann_results['fold_models'][0]
    demo_scaler = fann_results['fold_scalers'][0]

    visualize_single_image_classification(
        F_vector=asymmetry_results['F'],
        trained_model=demo_model,
        scaler=demo_scaler,
        image_name=image_name,
        class_labels=['Normal', 'Abnormal']
    )
else:
    print("FANN results or models not available. Please run the FANN classification cells first.")

In [ ]:
# ── Cell A: Load batch features ───────────────────────────────────────────────
X = np.load(FEATURES_PATH)          # shape (N, 21)
y = np.load(LABELS_PATH)            # shape (N,)   0=normal 1=abnormal

print(f"Feature matrix : X={X.shape},  y={y.shape}")
print(f"Normal={int((y==0).sum())}  Abnormal={int((y==1).sum())}")

# Diagnostic step
print("X shape:", X.shape, "  y:", dict(zip(*np.unique(y, return_counts=True))))
print("Any NaN:", np.isnan(X).any(), "  Any Inf:", np.isinf(X).any())
print("Feature ranges (min/max per column):")
for i in range(X.shape[1]):
    print(f"  F{i+1:02d}: [{X[:,i].min():.3f}, {X[:,i].max():.3f}]")

# Quick sanity check — no NaN / Inf
assert not np.any(np.isnan(X)), "NaN in features — check batch processing"
assert not np.any(np.isinf(X)), "Inf in features — check batch processing"


# ── Cell B: Run 5-fold cross-validation ───────────────────────────────────────
fann_results = run_fann_5fold(
    X             = X,
    y             = y,
    n_folds       = 5,
    epochs        = 2000,
    batch_size    = 32,
    learning_rate = 0.001,
    patience      = 150,
    random_state  = 42,
    results_dir   = 'fann_results',
    verbose       = True,
)


# ── Cell C: Visualise ─────────────────────────────────────────────────────────
visualize_fann_results(
    results     = fann_results,
    results_dir = 'fann_results',
    show        = True,
)